In [0]:

# ClinicalTrials.gov API에서 수집한 원본 JSON을 Databricks 환경으로 적재한다. 이후 모든 변환은 이 원본을 기준으로 수행한다.
file_path = "/Volumes/workspace/default/clinical-trial/clinical-trial-1000.json"

# 원본 JSON을 Spark DataFrame으로 로드한다. 이후 nested JSON 구조를 Spark 연산으로 처리할 수 있도록 준비하는 단계다.
raw_df = (
    spark.read
    .option("multiLine", True)
    .json(file_path)
)

# Spark가 추론한 source schema를 확인한다. struct/array 형태의 nested 필드를 파악해 이후 정규화 및 변환 전략을 설계한다.
raw_df.printSchema()



root
 |-- nextPageToken: string (nullable = true)
 |-- studies: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- annotationSection: struct (nullable = true)
 |    |    |    |-- annotationModule: struct (nullable = true)
 |    |    |    |    |-- unpostedAnnotation: struct (nullable = true)
 |    |    |    |    |    |-- unpostedEvents: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- date: string (nullable = true)
 |    |    |    |    |    |    |    |-- dateUnknown: boolean (nullable = true)
 |    |    |    |    |    |    |    |-- type: string (nullable = true)
 |    |    |    |    |    |-- unpostedResponsibleParty: string (nullable = true)
 |    |    |-- derivedSection: struct (nullable = true)
 |    |    |    |-- conditionBrowseModule: struct (nullable = true)
 |    |    |    |    |-- ancestors: array (nullable = true)
 |    |    |    |    |    |-- element: stru

In [0]:
from pyspark.sql import functions as F


# 수집된 study 개수를 검증해 API 응답이 기대한 규모로 정상 적재되었는지 확인한다.
raw_df.select(
    F.size("studies").alias("study_count"),
    "nextPageToken"
).show(truncate=False)

+-----------+---------------------------------------+
|study_count|nextPageToken                          |
+-----------+---------------------------------------+
|1000       |ZVNj7o2Elu8o3lpwWti8q-numo6QfJdpbPSs2Pg|
+-----------+---------------------------------------+



In [0]:
# 처음에는 JSON 전체가 하나의 row이고 그 안에 studies 배열 1,000개가 들어 있었습니다.
# explode()는 그 배열 안의 study를 하나씩 분리해 각 study를 개별 row로 만든 것입니다.
# nested data를 분석 가능한 row 단위로 펼치는 첫 normalization 단계입니다.

# 최상위 studies 배열을 explode하여 각 임상시험을 하나의 row로 변환한다. 이후 study 단위의 필드 선택과 정규화를 가능하게 한다.
studies_df = (
    raw_df
    .select(F.explode("studies").alias("study"))
)

print(studies_df.count())
display(studies_df.limit(5))

1000


study


In [0]:
# nested JSON 내부의 값을 실제 컬럼처럼 꺼내본 것입니다.
# nested raw data에서 업무적으로 의미 있는 필드가 어디에 있는지 mapping해본 단계입니다.
# nested source에서 핵심 필드를 추출해 source-to-target mapping을 확인한다. Silver 테이블 설계를 위한 사전 검증 단계다.

study_preview = studies_df.select(
    F.col("study.protocolSection.identificationModule.nctId").alias("nct_id"),
    F.col("study.protocolSection.identificationModule.briefTitle").alias("title"),
    F.col("study.protocolSection.designModule.studyType").alias("study_type"),
    F.col("study.protocolSection.statusModule.overallStatus").alias("status"),
    F.col("study.hasResults").alias("has_results")
)

display(study_preview.limit(20))

nct_id,title,study_type,status,has_results
NCT05281835,Follow-up of Elderly Subjects With Falls Recruited in the Geriatric Departments of Several French Hospitals,OBSERVATIONAL,ACTIVE_NOT_RECRUITING,false
NCT00629135,Moxifloxacin Plus Metronidazole Versus Piperacillin/Tazobactam for the Treatment of Patients With Intra-abdominal Abscesses,INTERVENTIONAL,COMPLETED,false
NCT04136535,Pemetrexed and Carboplatin With or Without Anlotinib Hydrochloride for Osimertinib-resistant Non-squamous NSCLC,INTERVENTIONAL,RECRUITING,false
NCT05380635,PK and ECG Determinations Following 8 Weeks of HyBryte Treatment for Cutaneous T-Cell Lymphoma,INTERVENTIONAL,COMPLETED,true
NCT04942535,STEP Together: An Effectiveness-Implementation Study of Social Incentives and Physical Activity,INTERVENTIONAL,COMPLETED,false
NCT01812135,Safety and Immunogenicity of EV71 Vaccine With or Without Aluminum Adjuvant in Infants,INTERVENTIONAL,WITHDRAWN,false
NCT05630235,Effects of CBD/CBD-A Oral Extract on Resting-state EEG and Neuropathic Pain Symptoms After SCI,INTERVENTIONAL,COMPLETED,false
NCT04885335,Evaluation of the Management of Acute Appendicitis Before Emergency Department in Children: a Prospective Study,OBSERVATIONAL,UNKNOWN,false
NCT03166735,Different Doses of BI 1467335 Compared to Placebo in Patients With Clinical Evidence of NASH,INTERVENTIONAL,COMPLETED,true
NCT01197235,Effect of Darbepoetin in Contrast-induced Nephropathy,INTERVENTIONAL,TERMINATED,false


In [0]:
# 원본 study 구조는 유지하고 source, ingestion time, source file 등의 metadata를 추가한다. 
# 데이터 lineage와 재현성을 확보하기 위한 Bronze Layer 구성이다.

bronze_df = (
    studies_df
    .select("study")
    .withColumn("source", F.lit("clinicaltrials.gov"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.lit("clinical-trial-1000.json"))
)

bronze_df.printSchema()

root
 |-- study: struct (nullable = true)
 |    |-- annotationSection: struct (nullable = true)
 |    |    |-- annotationModule: struct (nullable = true)
 |    |    |    |-- unpostedAnnotation: struct (nullable = true)
 |    |    |    |    |-- unpostedEvents: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- date: string (nullable = true)
 |    |    |    |    |    |    |-- dateUnknown: boolean (nullable = true)
 |    |    |    |    |    |    |-- type: string (nullable = true)
 |    |    |    |    |-- unpostedResponsibleParty: string (nullable = true)
 |    |-- derivedSection: struct (nullable = true)
 |    |    |-- conditionBrowseModule: struct (nullable = true)
 |    |    |    |-- ancestors: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |    |-- term: string (nullable = true)
 |    |    |  

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

spark.sql("""
CREATE SCHEMA IF NOT EXISTS clinical_trials
""")


# 정제 전 원본 study를 Delta Table로 저장해 지속적으로 조회·재처리 가능한 Bronze 데이터 계층을 구성한다.
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clinical_trials.bronze_studies")
)
# Databricks 환경에서 이후 증분 적재와 MERGE, schema 관리, 변경 이력 추적을 고려해 Bronze 데이터를 Delta Table로 저장했습니다. 단순 파일 저장보다 운영 가능한 데이터 파이프라인을 구성하기에 적합하다고 판단했습니다.

# CSV/JSON: 사람이 보기 쉬운 텍스트 형식
# Parquet: 분석용으로 효율적인 컬럼 기반 파일 형식
# Delta Lake: Parquet에 테이블 관리 기능을 더한 형식 Delta = Parquet 데이터 파일 + 트랜잭션 로그/테이블 관리 기능
# ACID transaction: 데이터 변경이 중간에 깨지지 않도록 보장하는 규칙

spark.sql("""
SELECT COUNT(*)
FROM clinical_trials.bronze_studies
""").show()

+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace|         default|
+-----------------+----------------+

+--------+
|COUNT(*)|
+--------+
|    1000|
+--------+

